# Fig. 2 — Spatial propagation fields


In [ ]:
# Setup
# Load the events and contact coordinates computed in Notebook 01.
import _setup
import numpy as np
from analysis import from_events as analysis
from analysis.src.topic5_interictal_direction_rose import fit_event_directions_3d
from plotting import computational as plot
from workflows.tutorial import load_events, TutorialOutput, save_fields

events, xyz, config = load_events()
out = TutorialOutput('fig2')


In [ ]:
# Fig. 2A: Contact geometry
# Each coordinate row corresponds to the same named contact in the event table.
out.show(plot.contact_geometry(xyz), 'fig2_A_geometry')


In [ ]:
# Fig. 2B: Rank-gradient directions
# Fit rank = intercept + beta dot xyz for each event.
# Larger ranks occur later, so beta / ||beta|| points from early to late.
# Group event directions by the timing-only sequence templates.
clustered = analysis.cluster_events(events)
ranks = np.where(events['participating'], events['ranks'], np.nan)
directions = fit_event_directions_3d(ranks, xyz)
out.show(plot.rank_directions(directions, clustered), 'fig2_B_rank_gradient')


In [ ]:
# Fig. 2B: Spatial information gain
# Train Timing and Timing+Space on separate recording blocks from the test data.
# Assign test events using ranks only, then score their directions on a common event set.
# Direction shuffling provides the null baseline for the gain.
directions, gain = analysis.direction_and_crossfit(events, xyz, n_null=1000, n_train_null=50)
out.show(plot.crossfit_scores(gain), 'fig2_B_crossfit')


In [ ]:
# Fig. 2C: Shared-plane fields
# After cross-validation, fit descriptive templates using all events.
# Project both templates onto the same TA plane and smooth earliness, -z(rank),
# with participation-weighted Gaussian kernels; leave weakly supported regions blank.
clustered = analysis.spatial_templates(events, xyz, directions)
field_record = analysis.template_fields(events, clustered, xyz, n_boot=200,
    subject_id=config['subject'], subject=config['subject'], dataset=config.get('dataset', 'raw_demo'))
field_a = analysis.illustrative_grid(events, clustered, xyz, 0)
field_b = analysis.illustrative_grid(events, clustered, xyz, 1, plane=field_a['plane'])
out.show(plot.field_panels([field_a, field_b],
    field_record['axis_pair'].get('strict_stability_pass', False)), 'fig2_C_fields')


In [ ]:
# Fig. 2D: Field similarity
# Correlate the two fields in the shared plane.
# Shuffle TB contact identities and rebuild its field to obtain the null distribution.
# Field correlation and the angle between propagation axes are different observables.
similarity = analysis.shared_field_similarity(field_record, n_perm=1000)
out.show(plot.shared_field_null(similarity), 'fig2_D_shared_field_null')


In [ ]:
# Save spatial results
# Save the cross-validation scores, projected fields, and permutation results.
save_fields(out, gain, clustered, field_record, field_a, field_b, similarity)
